# Arquiteturas de Domínio e Linhas de Produto (LPS) — Exemplo Executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ifpe-belojardim-es.github.io/guia-arquitetura-de-software/content/07-arquitetura-dominio-linhas-de-produto-reutilizacao/)

**Autores:**
- Maria Letícia de Sousa Barboza
- João Henrique Lopes de Araújo Freire
- Pedro Henrique Reis Xavier
- Wellison Danniel Soares Ponciano
- Caio Vinícius Santana Gomes
- Ruan Henrique Pereira dos Santos

**Licença:** CC BY-NC 4.0

---

Neste notebook vamos construir passo a passo uma família de **Lojas Web** usando
os conceitos de **Linha de Produtos de Software (LPS)**. Em vez de copiar e
colar o código para cada cliente, vamos ter um núcleo único e encaixar extras
como se fossem peças de montar.

```{mermaid}
%%| label: fig-nb-visao-geral
%%| fig-cap: "Fluxo do exemplo: a Engenharia de Domínio fornece regras e peças para a Engenharia de Aplicação derivar produtos válidos."
%%| fig-alt: "Diagrama de fluxo mostrando Modelo de Features, LojaBase e Decorators alimentando o Validador e a fábrica derivar_loja, que gera Loja Básica, Loja Express e Loja VIP."
%%{init: {'themeVariables': {'edgeLabelBackground': '#ffffff'}}}%%
flowchart LR
    classDef domain fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20,font-weight:bold
    classDef assets fill:#e1f5fe,stroke:#0277bd,stroke-width:2px,color:#01579b,font-weight:bold
    classDef app fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,color:#e65100,font-weight:bold
    classDef product fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#4a148c,font-weight:bold

    subgraph ED ["Engenharia de Domínio"]
        direction TB
        MF["Modelo de Features<br/>(Regras do que pode ou não)"]
        LB["Núcleo Comum<br/>(LojaBase)"]
        DEC["Extras Encaixáveis<br/>(Decorators)"]
    end

    subgraph EA ["Engenharia de Aplicação"]
        direction TB
        VAL["Validador<br/>(Porteiro)"]
        FAB["derivar_loja(...)<br/>(Fábrica)"]
        VAL --> FAB
    end

    P1["Loja Básica"]
    P2["Loja Express"]
    P3["Loja VIP"]

    MF -.->|"regras válidas"| VAL
    LB ==>|"núcleo"| FAB
    DEC ==>|"peças opcionais"| FAB
    FAB --> P1
    FAB --> P2
    FAB --> P3

    style ED fill:#f8fafc,stroke:#94a3b8,stroke-width:1.5px,color:#0f172a
    style EA fill:#f8fafc,stroke:#94a3b8,stroke-width:1.5px,color:#0f172a
    class MF domain
    class LB,DEC assets
    class VAL,FAB app
    class P1,P2,P3 product
```

**Regra principal:** primeiro o modelo valida se a combinação de recursos é
permitida; depois montamos a loja juntando o núcleo comum com os extras
escolhidos — sem copiar o projeto inteiro e sem encher o núcleo de `if`s por
cliente.

Roteiro:

1. Seção 2: o jeito certo — validar o Modelo de Features e derivar lojas com *Decorator*.
2. Seção 3: a violação — pular o validador e cair no antipadrão *clone-and-own*.
3. Seção 4: a medição — comparar combinações válidas, isolamento e custo de manutenção.

## 0. Preparação

Usamos apenas recursos que já vêm instalados no Python (`itertools`, `inspect`
e `sys`), então não é preciso instalar nenhuma biblioteca externa.

In [ ]:
from itertools import product  # gera todas as combinações possíveis de features
import inspect                 # usado na seção 4 para ler o código das classes
import sys

print("Versão do Python:", sys.version.split()[0])

## 1. O problema

Imagine que você desenvolveu uma **Loja Web**. No começo havia só um cliente,
mas logo surgiram dezenas de pedidos diferentes:

- Uma loja simples quer só **envio econômico**.
- Outra quer **envio rápido**, **carrinho de compras** e **preferências salvas**.
- Uma terceira vende para fora do país e quer **todos os envios** mais
  **programa de bônus**.

Se a equipe copiar e colar o código inteiro para cada cliente (*clone-and-own*),
em pouco tempo haverá dezenas de pastas separadas para cuidar. Quando aparecer
uma mudança comum — por exemplo, alterar a taxa de imposto ou corrigir um erro
no cálculo do pedido — será preciso abrir cópia por cópia à mão. Basta esquecer
uma cópia para um cliente passar a cobrar errado.

A solução da **Linha de Produtos de Software (LPS)** é dividir o sistema em:

1. **Um núcleo comum** com o que toda loja precisa ter (**Arquitetura de Domínio**).
2. **Peças variáveis** para os recursos que mudam de um cliente para outro.
3. **Um Modelo de Features** que garante que nenhuma combinação proibida seja montada.

## 2. Implementação seguindo o estilo

### 2.1 O Modelo de Features da Loja Web (o "porteiro")

Antes de escrever as classes da loja, definimos as regras da família no
**Modelo de Features**:

- **Envio (obrigatória, grupo `OR`):** toda loja precisa ter **pelo menos 1**
  tipo de envio entre `Economico`, `Rapido` e `Exterior`. Não existe loja sem
  forma de entrega.
- **Features opcionais:** `Carrinho`, `Bonus` e `Preferencias` podem estar
  ligadas ou desligadas em cada loja.
- **Restrição entre ramos (`Bônus requer Carrinho`):** quando ativada, impede
  que uma loja tenha programa de `Bonus` sem ter `Carrinho`.

A função `validar_configuracao` funciona como um porteiro: ela checa essas
regras e avisa na hora se a combinação é válida ou proibida.

```{mermaid}
%%| label: fig-nb-feature-model
%%| fig-cap: "Modelo de Features da Loja Web usado pelo validador em Python."
%%| fig-alt: "Árvore do modelo de features da Loja Web com Envio obrigatório em grupo OR (Economico, Rapido, Exterior), opcionais Carrinho, Bonus e Preferencias, e seta de restrição Bonus requer Carrinho."
%%{init: {'themeVariables': {'edgeLabelBackground': '#ffffff', 'primaryTextColor': '#0f172a', 'lineColor': '#475569'}}}%%
flowchart TD
    classDef root fill:#e2e8f0,stroke:#1e293b,stroke-width:2px,color:#0f172a,font-weight:bold
    classDef mand fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20,font-weight:bold
    classDef opt fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,color:#e65100,font-weight:bold
    classDef grp fill:#e1f5fe,stroke:#0277bd,stroke-width:1.5px,color:#01579b,font-weight:bold

    R["Loja Web"]
    E["Envio"]
    OR(["OR (pelo menos 1)"])
    EC["Economico"]
    RA["Rapido"]
    EX["Exterior"]
    CA["Carrinho"]
    BO["Bonus"]
    PR["Preferencias"]

    R -->|"obrigatória"| E
    R -.->|"opcional"| CA
    R -.->|"opcional"| BO
    R -.->|"opcional"| PR

    E --- OR
    OR --> EC
    OR --> RA
    OR --> EX

    BO -.->|"requer"| CA

    linkStyle default color:#0f172a
    class R root
    class E mand
    class CA,BO,PR opt
    class OR,EC,RA,EX grp
```

In [ ]:
class ConfiguracaoInvalidaError(Exception):
    """Erro lançado quando a escolha de features viola o Modelo de Features."""


ENVIOS_DISPONIVEIS = ("Economico", "Rapido", "Exterior")
OPCIONAIS_DISPONIVEIS = ("Carrinho", "Bonus", "Preferencias")


def validar_configuracao(envios, opcionais, exigir_carrinho_para_bonus=False):
    """Confere se as features escolhidas formam um produto válido.

    Devolve (True, mensagem) se tudo estiver certo, ou (False, motivo) se violar
    alguma regra do modelo.
    """
    # 1. Garante que ninguém pediu uma feature que não existe
    for envio in envios:
        if envio not in ENVIOS_DISPONIVEIS:
            return False, f"Envio desconhecido: '{envio}'"

    for extra in opcionais:
        if extra not in OPCIONAIS_DISPONIVEIS:
            return False, f"Feature opcional desconhecida: '{extra}'"

    # 2. Grupo OR obrigatório: precisa escolher pelo menos 1 envio
    if len(envios) == 0:
        return False, "Envio é obrigatório: escolha ao menos 1 (Economico, Rapido ou Exterior)."

    # 3. Restrição entre ramos: Bonus requer Carrinho
    if exigir_carrinho_para_bonus and "Bonus" in opcionais and "Carrinho" not in opcionais:
        return False, "Regra violada: 'Bonus' requer 'Carrinho'."

    return True, "Configuração válida"


# Testando 3 pedidos de configuração no porteiro
exemplos = [
    (["Economico"], ["Carrinho"]),  # válida
    ([], ["Carrinho"]),             # inválida: nenhum método de envio
    (["Rapido"], ["Bonus"]),          # inválida quando Bonus requer Carrinho
]

for envios, opcionais in exemplos:
    valido, mensagem = validar_configuracao(envios, opcionais, exigir_carrinho_para_bonus=True)
    situacao = "ACEITA" if valido else "BLOQUEADA"
    print(f"[{situacao}] envios={envios}, opcionais={opcionais} -> {mensagem}")

### 2.2 Comprovando os 56 produtos válidos na prática

Com 6 escolhas de "liga ou desliga" (`Economico`, `Rapido`, `Exterior`,
`Carrinho`, `Bonus`, `Preferencias`), existem $2^6 = 64$ combinações possíveis.

Porém, como toda loja precisa de pelo menos 1 tipo de envio, as opções de envio
validam apenas $2^3 - 1 = 7$ escolhas (a escolha "nenhum envio" é proibida).
Multiplicando pelas 3 features opcionais ($2 \times 2 \times 2 = 8$), temos:

$$7 \times 2 \times 2 \times 2 = 56 \text{ produtos válidos}$$

E quando ativamos também a restrição `Bonus requer Carrinho`, os 14 casos que
têm `Bonus` sem `Carrinho` também são bloqueados, sobrando **42 produtos
válidos**. Vamos fazer o Python testar as 64 combinações e conferir esses números.

In [ ]:
todas_combinacoes = []
for escolhas in product([False, True], repeat=6):
    envios = []
    for i in range(3):
        if escolhas[i]:
            envios.append(ENVIOS_DISPONIVEIS[i])

    opcionais = []
    for i in range(3):
        if escolhas[i + 3]:
            opcionais.append(OPCIONAIS_DISPONIVEIS[i])

    todas_combinacoes.append((envios, opcionais))

validas_basico = []
invalidas_basico = []
for envios, opcionais in todas_combinacoes:
    ok, motivo = validar_configuracao(envios, opcionais, exigir_carrinho_para_bonus=False)
    if ok:
        validas_basico.append((envios, opcionais))
    else:
        invalidas_basico.append((envios, opcionais, motivo))

validas_com_restricao = []
for envios, opcionais in todas_combinacoes:
    ok, _ = validar_configuracao(envios, opcionais, exigir_carrinho_para_bonus=True)
    if ok:
        validas_com_restricao.append((envios, opcionais))

print(f"Combinações possíveis no total (2^6): {len(todas_combinacoes)}")
print(f"Modelo básico (só regra de envio):    {len(validas_basico)} válidas | {len(invalidas_basico)} bloqueadas")
print(f"Com regra 'Bonus requer Carrinho':    {len(validas_com_restricao)} válidas | {len(todas_combinacoes) - len(validas_com_restricao)} bloqueadas")

assert len(validas_basico) == 56
assert len(validas_com_restricao) == 42
print("\nExemplo bloqueado pelo validador:", invalidas_basico[0])

### 2.3 Núcleo comum (`LojaBase`) e extras encaixáveis (*Decorator*)

Agora vamos para a implementação das lojas. Pense em um celular básico e nas
capinhas/acessórios que você encaixa nele:

- **`LojaBase`** é o núcleo comum da Arquitetura de Domínio. Ela sabe o que
  **toda** loja precisa fazer: somar o subtotal dos produtos, calcular o imposto
  padrão (`5%`) e cobrar o frete de um método de envio permitido. Ela **não
  conhece** `Carrinho`, `Bonus` nem `Preferencias`.
- **`LojaDecorator`** é o molde para os extras: ele recebe uma loja por dentro e
  repassa o pedido para ela.
- **`CarrinhoDecorator`**: dá **R$ 10,00 de desconto** quando o pedido tem 2 ou
  mais itens.
- **`PreferenciasDecorator`**: dá **10% de desconto no frete** usando as
  preferências salvas do cliente.
- **`BonusDecorator`**: calcula **1 ponto de bônus a cada R$ 10,00** do total
  final.

Como cada decorador tem o mesmo método `finalizar_pedido`, podemos empilhar
quantos quisermos sem alterar uma única linha da `LojaBase`:

```{mermaid}
%%| label: fig-nb-decorator
%%| fig-cap: "Empilhamento dos decoradores ao redor de LojaBase na derivação da Loja VIP."
%%| fig-alt: "Diagrama de fluxo horizontal mostrando LojaBase sendo estendida em camadas por CarrinhoDecorator, PreferenciasDecorator e BonusDecorator até formar a Loja VIP pronta."
flowchart LR
    classDef core fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20,font-weight:bold
    classDef deco fill:#e1f5fe,stroke:#0277bd,stroke-width:2px,color:#01579b,font-weight:bold
    classDef prod fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#4a148c,font-weight:bold

    B["LojaBase<br/>(subtotal + imposto + frete)"]
    D1["CarrinhoDecorator<br/>(-R$ 10 com 2+ itens)"]
    D2["PreferenciasDecorator<br/>(-10% no frete)"]
    D3["BonusDecorator<br/>(pontos no total final)"]
    VIP["Loja VIP pronta"]

    B -->|"+ Carrinho"| D1 -->|"+ Preferencias"| D2 -->|"+ Bonus"| D3 --> VIP

    class B core
    class D1,D2,D3 deco
    class VIP prod
```


In [ ]:
class LojaBase:
    """Núcleo comum (Arquitetura de Domínio): envio e imposto padrão."""

    TAXA_IMPOSTO = 0.05  # 5% de imposto comum a todas as lojas da família
    TABELA_FRETE = {
        "Economico": 15.0,
        "Rapido": 30.0,
        "Exterior": 60.0,
    }

    def __init__(self, nome, envios_permitidos):
        self.nome = nome
        self.envios_permitidos = list(envios_permitidos)

    def finalizar_pedido(self, itens, tipo_envio):
        if tipo_envio not in self.envios_permitidos:
            raise ValueError(
                f"A loja '{self.nome}' não oferece envio '{tipo_envio}'. "
                f"Opções disponíveis: {self.envios_permitidos}"
            )

        subtotal = 0.0
        for _, preco in itens:
            subtotal = subtotal + preco
        imposto = round(subtotal * self.TAXA_IMPOSTO, 2)
        frete = self.TABELA_FRETE[tipo_envio]
        total = round(subtotal + imposto + frete, 2)

        return {
            "loja": self.nome,
            "itens": len(itens),
            "envio": tipo_envio,
            "subtotal": round(subtotal, 2),
            "imposto": imposto,
            "frete": frete,
            "desconto": 0.0,
            "pontos_bonus": 0,
            "total": total,
            "extras": [],
        }


class LojaDecorator:
    """Base dos decoradores: envolve uma loja e repassa a chamada."""

    def __init__(self, loja):
        self.loja = loja
        self.nome = loja.nome
        self.envios_permitidos = loja.envios_permitidos

    def finalizar_pedido(self, itens, tipo_envio):
        return self.loja.finalizar_pedido(itens, tipo_envio)


class CarrinhoDecorator(LojaDecorator):
    """Feature 'Carrinho': dá R$ 10,00 de desconto para 2 ou mais itens."""

    def finalizar_pedido(self, itens, tipo_envio):
        resumo = super().finalizar_pedido(itens, tipo_envio)
        if resumo["itens"] >= 2:
            resumo["desconto"] = round(resumo["desconto"] + 10.0, 2)
            resumo["total"] = round(resumo["total"] - 10.0, 2)
        resumo["extras"].append("Carrinho")
        return resumo


class PreferenciasDecorator(LojaDecorator):
    """Feature 'Preferencias': aplica 10% de desconto no valor do frete."""

    def finalizar_pedido(self, itens, tipo_envio):
        resumo = super().finalizar_pedido(itens, tipo_envio)
        desconto_frete = round(resumo["frete"] * 0.10, 2)
        resumo["frete"] = round(resumo["frete"] - desconto_frete, 2)
        resumo["total"] = round(resumo["total"] - desconto_frete, 2)
        resumo["extras"].append("Preferencias")
        return resumo


class BonusDecorator(LojaDecorator):
    """Feature 'Bonus': gera 1 ponto a cada R$ 10,00 do valor total final."""

    def finalizar_pedido(self, itens, tipo_envio):
        resumo = super().finalizar_pedido(itens, tipo_envio)
        resumo["pontos_bonus"] = int(resumo["total"] // 10)
        resumo["extras"].append("Bonus")
        return resumo

### 2.4 Derivando 3 lojas diferentes (Engenharia de Aplicação)

Na **Engenharia de Aplicação**, nós não criamos arquivos novos para cada cliente.
Passamos apenas a lista de *features* para a função `derivar_loja(...)`:

1. Ela chama o validador do Modelo de Features (se houver erro, recusa na hora).
2. Cria a `LojaBase` com os envios escolhidos.
3. Encaixa os decoradores correspondentes às *features* opcionais ligadas.

Veja como derivamos **3 lojas reais** diferentes com a mesma plataforma.

In [ ]:
DECORADORES_OPCIONAIS = {
    "Carrinho": CarrinhoDecorator,
    "Preferencias": PreferenciasDecorator,
    "Bonus": BonusDecorator,  # fica por fora para contar pontos após os descontos
}


def derivar_loja(nome, envios, opcionais, exigir_carrinho_para_bonus=True):
    """Engenharia de Aplicação: valida as features e monta o produto."""
    valido, motivo = validar_configuracao(
        envios, opcionais, exigir_carrinho_para_bonus=exigir_carrinho_para_bonus
    )
    if not valido:
        raise ConfiguracaoInvalidaError(motivo)

    produto = LojaBase(nome=nome, envios_permitidos=envios)
    for feature, classe_decorator in DECORADORES_OPCIONAIS.items():
        if feature in opcionais:
            produto = classe_decorator(produto)
    return produto


itens_compra = [("Teclado", 150.0), ("Mouse", 50.0)]  # subtotal = R$ 200,00

loja_basica = derivar_loja("Loja Básica", ["Economico"], [])
loja_express = derivar_loja("Loja Express", ["Economico", "Rapido"], ["Carrinho", "Preferencias"])
loja_vip = derivar_loja(
    "Loja VIP",
    ["Economico", "Rapido", "Exterior"],
    ["Carrinho", "Bonus", "Preferencias"],
)

for loja, envio in [(loja_basica, "Economico"), (loja_express, "Rapido"), (loja_vip, "Exterior")]:
    resumo = loja.finalizar_pedido(itens_compra, envio)
    print(
        f"{resumo['loja']:13} | envio={resumo['envio']:9} | "
        f"total=R$ {resumo['total']:6.2f} | pontos={resumo['pontos_bonus']:2} | "
        f"extras={resumo['extras']}"
    )

# Tentando derivar uma loja proibida (Bonus sem Carrinho)
try:
    derivar_loja("Loja Proibida", ["Rapido"], ["Bonus"])
except ConfiguracaoInvalidaError as erro:
    print("\nDerivação bloqueada corretamente:", erro)

### 2.5 O núcleo funciona sozinho (regra de ouro)

Em uma LPS bem feita, o código comum (`LojaBase`) **nunca** pode assumir a
existência de uma feature opcional. Vamos testar a `LojaBase` sozinha, sem
nenhum decorador ao redor.

In [ ]:
nucleo = LojaBase("Loja Isolada", ["Economico"])
resumo_nucleo = nucleo.finalizar_pedido([("Caderno", 100.0)], "Economico")

assert resumo_nucleo["subtotal"] == 100.0
assert resumo_nucleo["imposto"] == 5.0
assert resumo_nucleo["frete"] == 15.0
assert resumo_nucleo["total"] == 120.0
assert resumo_nucleo["extras"] == []
print("Teste passou: LojaBase funciona sozinha e não depende de nenhum extra.")

## 3. A violação

Agora vamos quebrar as regras de propósito para ver o que dá errado na prática:

1. **Pular o validador do Modelo de Features:** instanciar `LojaBase` e
   decoradores diretamente na mão, criando produtos impossíveis (uma loja sem
   nenhum método de envio, e outra com `Bonus` sem `Carrinho`).
2. **Cair no antipadrão *Clone-and-Own*:** copiar e colar o código da loja para
   cada cliente (`LojaCloneA`, `LojaCloneB`, `LojaCloneC`). Quando a taxa de
   imposto comum precisa subir de `5%` para `8%`, a equipe atualiza dois clones
   e esquece o terceiro — gerando **divergência silenciosa** nos preços.

```{mermaid}
%%| label: fig-nb-violacao
%%| fig-cap: "Comparação da manutenção ao mudar o imposto de 5% para 8%: 1 mudança na LPS contra N edições manuais no clone-and-own."
%%| fig-alt: "Diagrama comparando a LPS, onde alterar LojaBase para 8% atualiza todas as lojas, com o clone-and-own, onde esquecer LojaCloneC em 5% gera divergência."
%%{init: {'themeVariables': {'edgeLabelBackground': '#ffffff'}}}%%
flowchart LR
    classDef ok fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20,font-weight:bold
    classDef err fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#b71c1c,font-weight:bold
    classDef neutral fill:#f8fafc,stroke:#64748b,stroke-width:1.5px,color:#1e293b

    subgraph LPS ["LPS: 1 mudança"]
        direction TB
        N["LojaBase (8%)"]
        L1["Loja 1: 8%"]
        L2["Loja 2: 8%"]
        L3["Loja 3: 8%"]
        N --> L1
        N --> L2
        N --> L3
    end

    subgraph CLONE ["Clone-and-Own: N edições"]
        direction TB
        CA["LojaCloneA: 8% (atualizada)"]
        CB["LojaCloneB: 8% (atualizada)"]
        CC["LojaCloneC: 5% (ESQUECIDA!)"]
    end

    style LPS fill:#f1f8e9,stroke:#81c784,stroke-width:1.5px,color:#1b5e20
    style CLONE fill:#fff5f5,stroke:#e57373,stroke-width:1.5px,color:#b71c1c
    class N,L1,L2,L3 ok
    class CA,CB neutral
    class CC err
```


In [ ]:
# VIOLAÇÃO 1: montar lojas na mão, pulando o validador do Modelo de Features
loja_sem_envio = LojaBase("Loja Quebrada", envios_permitidos=[])
loja_bonus_sem_carrinho = BonusDecorator(LojaBase("Loja Sem Carrinho", ["Economico"]))

print("Violação 1a -> loja criada sem envio:", loja_sem_envio.envios_permitidos)
try:
    loja_sem_envio.finalizar_pedido([("Livro", 80.0)], "Economico")
except ValueError as erro:
    print("  Só descobrimos o problema tarde demais, na compra:", erro)

resumo_proibido = loja_bonus_sem_carrinho.finalizar_pedido([("Livro", 80.0)], "Economico")
print(
    "Violação 1b -> deu bônus mesmo sem carrinho:",
    resumo_proibido["extras"],
    "| pontos =",
    resumo_proibido["pontos_bonus"],
)


# VIOLAÇÃO 2 (ANTIPADRÃO): Clone-and-Own — um arquivo/classe copiado por cliente
class LojaCloneA:
    TAXA_IMPOSTO = 0.08  # atualizado de 5% para 8%

    def calcular_total(self, subtotal, frete):
        return round(subtotal + subtotal * self.TAXA_IMPOSTO + frete, 2)


class LojaCloneB:
    TAXA_IMPOSTO = 0.08  # atualizado de 5% para 8%

    def calcular_total(self, subtotal, frete):
        return round(subtotal + subtotal * self.TAXA_IMPOSTO + frete, 2)


class LojaCloneC:
    TAXA_IMPOSTO = 0.05  # ESQUECERAM de atualizar este clone!

    def calcular_total(self, subtotal, frete):
        return round(subtotal + subtotal * self.TAXA_IMPOSTO + frete, 2)


print("\nViolação 2 (Clone-and-Own após mudança do imposto de 5% para 8%):")
print("  Clone A (subtotal 200, frete 15): R$", LojaCloneA().calcular_total(200.0, 15.0))
print("  Clone B (subtotal 200, frete 15): R$", LojaCloneB().calcular_total(200.0, 15.0))
print("  Clone C (subtotal 200, frete 15): R$", LojaCloneC().calcular_total(200.0, 15.0), "<- DIVERGÊNCIA SILENCIOSA!")

## 4. Consequência mensurável

Em vez de apenas dizer que a LPS é mais segura e fácil de manter, vamos medir
três números objetivos:

1. **Isolamento do núcleo:** contamos quantas vezes as features opcionais
   (`Carrinho`, `Bonus`, `Preferencias`) aparecem no código de `LojaBase` (deve ser **0**).
2. **Configurações barradas pelo validador:** quantas combinações inválidas foram
   interceptadas antes de gerar código (8 sem envio; 22 quando `Bonus requer Carrinho`).
3. **Custo de corrigir uma regra comum em 5 lojas:** alteramos o imposto de `5%`
   para `8%` na LPS (1 único lugar) e comparamos com a manutenção em 5 clones
   onde 1 clone foi esquecido.

In [ ]:
def contar_citacoes_opcionais(classe):
    """Conta quantas vezes features opcionais aparecem nos métodos da classe."""
    total = 0
    for metodo in vars(classe).values():
        if callable(metodo):
            codigo = inspect.getsource(metodo)
            for feat in OPCIONAIS_DISPONIVEIS:
                total = total + codigo.count(feat)
    return total


citacoes_nucleo = contar_citacoes_opcionais(LojaBase)

# Simulando a correção da taxa de imposto (5% -> 8%) em 5 lojas derivadas na LPS
LojaBase.TAXA_IMPOSTO = 0.08  # 1 único ponto alterado na plataforma

lojas_lps = [
    derivar_loja("Cliente 1", ["Economico"], []),
    derivar_loja("Cliente 2", ["Economico", "Rapido"], ["Carrinho"]),
    derivar_loja("Cliente 3", ["Rapido"], ["Preferencias"]),
    derivar_loja("Cliente 4", ["Economico", "Exterior"], ["Carrinho", "Bonus"]),
    derivar_loja("Cliente 5", ["Economico", "Rapido", "Exterior"], ["Carrinho", "Bonus", "Preferencias"]),
]

divergencias_lps = 0
for loja in lojas_lps:
    resumo = loja.finalizar_pedido([("Item", 200.0)], loja.envios_permitidos[0])
    if resumo["imposto"] != 16.0:  # 8% de 200 = 16.0
        divergencias_lps = divergencias_lps + 1

LojaBase.TAXA_IMPOSTO = 0.05  # restaura 5% caso o aluno reexecute a seção 2

# Comparando com 5 clones manuais (um deles ficou desatualizado em 5%)
clones = [LojaCloneA(), LojaCloneB(), LojaCloneC(), LojaCloneA(), LojaCloneB()]
divergencias_clones = 0
for clone in clones:
    total = clone.calcular_total(200.0, 15.0)
    if total != 231.0:  # 200 + 16 (8%) + 15 (frete) = 231.0
        divergencias_clones = divergencias_clones + 1

print(f"Citações a features opcionais dentro de LojaBase: {citacoes_nucleo}")
print(f"Combinações possíveis: {len(todas_combinacoes)} | "
      f"Válidas (básico): {len(validas_basico)} | Interceptadas: {len(invalidas_basico)}")
print(f"Com 'Bonus requer Carrinho': {len(validas_com_restricao)} válidas | "
      f"Interceptadas: {len(todas_combinacoes) - len(validas_com_restricao)}\n")

print(f"{'Abordagem':24} | {'Pontos alterados':16} | {'Divergências em 5 lojas'}")
print("-" * 70)
print(f"{'LPS (Domínio+Decorator)':24} | {1:<16} | {divergencias_lps} de {len(lojas_lps)}")
print(f"{'Clone-and-Own':24} | {len(clones):<16} | {divergencias_clones} de {len(clones)}")

## 5. Conclusão

Os números resumem por que uma **Linha de Produtos de Software** organiza
melhor o reuso em famílias de sistemas:

- O **Modelo de Features** impede que lojas inválidas sejam montadas, garantindo
  apenas as **56** combinações válidas do modelo básico (ou **42** com
  `Bônus requer Carrinho`).
- A **`LojaBase`** tem **0** citações às features opcionais, e os **Decorators**
  encaixam cada variação de forma isolada.
- Corrigir uma regra comum exige editar **1 único ponto** na plataforma e
  regenerar os *N* produtos, evitando as *N* alterações manuais e o risco de
  divergência silenciosa do *clone-and-own*.

Como visto nos *trade-offs* do tópico, esse investimento inicial vale muito a
pena quando a família cresce para dezenas de variantes; para 2 ou 3 protótipos
rápidos, copiar código pode ser mais simples no começo.

---

### Referências

[1] K. Pohl, G. Böckle e F. van der Linden, *Software Product Line Engineering: Foundations, Principles, and Techniques*. Springer, 2005.

[2] S. Apel, D. Batory, C. Kästner e G. Saake, *Feature-Oriented Software Product Lines: Concepts and Implementation*. Springer, 2013.

[3] K. C. Kang et al., *Feature-Oriented Domain Analysis (FODA) Feasibility Study*. SEI/CMU, CMU/SEI-90-TR-21, 1990.

[4] E. Gamma, R. Helm, R. Johnson e J. Vlissides, *Design Patterns: Elements of Reusable Object-Oriented Software*. Addison-Wesley, 1994.

[5] J. Krüger e T. Berger, "An Empirical Analysis of the Costs of Clone- and Platform-Oriented Software Reuse", *ESEC/FSE*, ACM, 2020.

---

Conteúdo sob CC BY-NC 4.0 — uso livre com crédito. Código sob MIT.